# Experiment 7 — Dimensionality reduction — PCA, SVD and LDA

## Run directly in Jupyter

- Keep the extracted `datasets` folder beside the `notebooks` folder.
- Run every cell from top to bottom with **Shift + Enter**, or choose **Run → Run All Cells**.
- No manual input or internet download is required.
- If imports are missing, run `%pip install numpy pandas matplotlib seaborn scikit-learn` in a code cell, then rerun the failed cell.

**Dataset:** `datasets/sample_titanic.csv` is an included synthetic teaching dataset, not the official Titanic passenger dataset. Scores are illustrative.

In [ ]:
# Load the included sample dataset. No internet or manual input is needed.
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

possible_paths = [
    Path("sample_titanic.csv"),
    Path("../datasets/sample_titanic.csv"),
    Path("datasets/sample_titanic.csv"),
]
csv_file = next((p for p in possible_paths if p.exists()), None)
if csv_file is None:
    raise FileNotFoundError(
        "sample_titanic.csv not found. Keep the datasets folder beside notebooks."
    )

df = pd.read_csv(csv_file)
df = df.rename(columns={
    "PassengerId": "passenger_id", "Survived": "survived", "Pclass": "pclass",
    "Name": "name", "Sex": "sex", "Age": "age", "SibSp": "sibsp",
    "Parch": "parch", "Fare": "fare", "Embarked": "embarked", "Cabin": "cabin"
})
print("Loaded sample dataset:", csv_file)
print("Dataset shape:", df.shape)
display(df.head())


## Aim
Apply PCA and Truncated SVD for unsupervised reduction and LDA for supervised reduction.

## 1. Prepare numeric features

In [ ]:
features=["pclass","age","sibsp","parch","fare"]
X=df[features].copy()
for col in features: X[col]=X[col].fillna(X[col].median())
y=df["survived"]
from sklearn.preprocessing import StandardScaler
X_scaled=StandardScaler().fit_transform(X)
print("Input shape:",X_scaled.shape)


## 2. Apply PCA, SVD and LDA

In [ ]:
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
pca=PCA(n_components=2)
X_pca=pca.fit_transform(X_scaled)
print("PCA shape:",X_pca.shape)
print("PCA explained variance ratio:",pca.explained_variance_ratio_.round(4))
print("Total PCA explained variance:",round(pca.explained_variance_ratio_.sum(),4))
svd=TruncatedSVD(n_components=2,random_state=42)
X_svd=svd.fit_transform(X_scaled)
print("\nSVD shape:",X_svd.shape)
print("SVD explained variance ratio:",svd.explained_variance_ratio_.round(4))
# There are two target classes, so LDA can produce at most one component.
lda=LinearDiscriminantAnalysis(n_components=1)
X_lda=lda.fit_transform(X_scaled,y)
print("\nLDA shape:",X_lda.shape)


## 3. Visualize PCA

In [ ]:
y_array=y.to_numpy()
plt.figure(figsize=(8,5))
for label,name in [(0,"Not Survived"),(1,"Survived")]:
    mask=y_array==label
    plt.scatter(X_pca[mask,0],X_pca[mask,1],label=name,alpha=0.65,s=25)
plt.xlabel("Principal Component 1"); plt.ylabel("Principal Component 2")
plt.title("PCA Projection of Titanic Features"); plt.legend(); plt.show()


## Expected output
PCA/SVD/LDA shapes, explained variance values, and a PCA scatter plot. LDA uses the target labels.